In [ ]:
import json
import pandas as pd
from pathlib import Path
import re
import numpy as np

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 120

# Yelp Processed Data — EDA
## Exploration of the processed parquets.

In [ ]:
DATA_DIR = "../data/raw/extracted/"
PROCESSED_DIR = "../data/processed/"

In [ ]:
restaurants = pd.read_parquet(f"{PROCESSED_DIR}/restaurants.parquet")
reviews = pd.read_parquet(f"{PROCESSED_DIR}/reviews.parquet")
users = pd.read_parquet(f"{PROCESSED_DIR}/users.parquet")

# These might not exist yet — load if available
try:
    checkins = pd.read_parquet(f"{PROCESSED_DIR}/checkin_profiles.parquet")
    print(f"Checkin profiles: {len(checkins):,}")
except FileNotFoundError:
    checkins = None
    print("No checkin_profiles.parquet found")

try:
    tips = pd.read_parquet(f"{PROCESSED_DIR}/tips.parquet")
    print(f"Tips: {len(tips):,}")
except FileNotFoundError:
    tips = None
    print("No tips.parquet found")

print(f"Restaurants: {len(restaurants):,}")
print(f"Reviews:     {len(reviews):,}")
print(f"Users:       {len(users):,}")

In [ ]:
# All categories on your 43,552 restaurants
all_cats = restaurants["categories"].explode().str.strip().str.lower()
cat_counts = all_cats.value_counts()
print(f"{cat_counts.nunique()} unique categories\n")
print(cat_counts.head(201))

In [ ]:
print(cat_counts.to_string())

### 2. Restaurants

In [ ]:
# -- Top 15 cities --
city_counts = (restaurants.groupby(["city", "state"]).size()
               .sort_values(ascending=False).head(15))
fig, ax = plt.subplots(figsize=(10, 5))
city_counts.plot.barh(ax=ax, color=list(sns.color_palette("viridis", len(city_counts))))
ax.set_xlabel("Restaurant Count")
ax.set_title("Top 15 Cities by Restaurant Count")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# -- Top 20 categories --
from collections import Counter
cat_counter = Counter()
for cats in restaurants["categories"].dropna():
    if isinstance(cats, (list, np.ndarray)):
        cat_counter.update(c.strip() for c in cats)
    elif isinstance(cats, str):
        cat_counter.update(c.strip() for c in cats.split(","))

top_cats = pd.DataFrame(cat_counter.most_common(20), columns=["category", "count"])
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_cats, y="category", x="count", ax=ax)
ax.set_title("Top 20 Restaurant Categories")
ax.set_xlabel("Count")
plt.tight_layout()
plt.show()

# -- Star rating distribution --
fig, ax = plt.subplots(figsize=(7, 4))
restaurants["stars"].value_counts().sort_index().plot.bar(ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Restaurant Star Rating Distribution")
ax.set_xlabel("Stars")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### 3. Reviews

In [ ]:
# -- Review star distribution --
fig, ax = plt.subplots(figsize=(7, 4))
reviews["stars"].value_counts().sort_index().plot.bar(
    ax=ax, color=list(sns.color_palette("coolwarm", 5)), edgecolor="white"
)
ax.set_title("Review Star Distribution")
ax.set_xlabel("Stars"); ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

# -- Reviews by day of week --
dow_map = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
dow_counts = reviews["day_of_week"].value_counts().sort_index().rename(index=dow_map)
fig, ax = plt.subplots(figsize=(7, 4))
dow_counts.plot.bar(ax=ax, color="coral", edgecolor="white")
ax.set_title("Reviews by Day of Week")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

# -- Review volume over time --
reviews_monthly = reviews.set_index("date").resample("M").size()
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(reviews_monthly.index, reviews_monthly.values, alpha=0.4, color="steelblue")
ax.plot(reviews_monthly.index, reviews_monthly.values, color="steelblue", linewidth=1)
ax.set_title("Monthly Review Volume (2005–2020)")
ax.set_ylabel("Reviews")
plt.tight_layout()
plt.show()

# -- Reviews per user (log-scale histogram) --
reviews_per_user = reviews.groupby("user_id").size()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(reviews_per_user.clip(upper=20), bins=range(1, 22), color="teal", edgecolor="white")
axes[0].set_xlim(1, 21)
axes[0].set_xticks(range(1, 21))
axes[0].set_title("Reviews per User (clipped at 20)")
axes[0].set_xlabel("Review Count"); axes[0].set_ylabel("Users")

reviews_per_biz = reviews.groupby("business_id").size()
axes[1].hist(reviews_per_biz.clip(upper=200), bins=50, color="salmon", edgecolor="white")
axes[1].set_title("Reviews per Restaurant (clipped at 200)")
axes[1].set_xlabel("Review Count"); axes[1].set_ylabel("Restaurants")

plt.tight_layout()
plt.show()

# -- Sparsity callout --
n_users = reviews["user_id"].nunique()
n_items = reviews["business_id"].nunique()
n_interactions = len(reviews)
sparsity = 1 - n_interactions / (n_users * n_items)
print(f"Users: {n_users:,}  |  Items: {n_items:,}  |  Interactions: {n_interactions:,}")
print(f"Sparsity: {sparsity:.4%}")


### 4. Users

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Review count distribution (log scale)
axes[0].hist(users["review_count"].clip(upper=100), bins=50, color="mediumpurple", edgecolor="white")
axes[0].set_title("User Review Counts (clipped at 100)")
axes[0].set_xlabel("Reviews"); axes[0].set_ylabel("Users")

# Average star distribution
axes[1].hist(users["average_stars"], bins=30, color="goldenrod", edgecolor="white")
axes[1].set_title("User Average Star Ratings")
axes[1].set_xlabel("Avg Stars"); axes[1].set_ylabel("Users")

plt.tight_layout()
plt.show()

### 5. Checkin Profiles

In [ ]:
if checkins is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Peak hour
    checkins["peak_hour"].value_counts().sort_index().plot.bar(
        ax=axes[0], color="steelblue", edgecolor="white"
    )
    axes[0].set_title("Peak Checkin Hour")
    axes[0].set_xlabel("Hour (0–23)")

    # Weekend ratio
    axes[1].hist(checkins["weekend_ratio"].dropna(), bins=30, color="coral", edgecolor="white")
    axes[1].set_title("Weekend Ratio Distribution")
    axes[1].set_xlabel("Weekend Ratio")

    # Total checkins (log scale)
    axes[2].hist(checkins["total_checkins"].clip(upper=200), bins=50, color="teal", edgecolor="white")
    axes[2].set_title("Total Checkins per Restaurant")
    axes[2].set_xlabel("Checkins (clipped at 200)")

    plt.tight_layout()
    plt.show()

### Checkin-to-Review Match Quality

In [ ]:
# -- Match rate by restaurant --
if "checkin_ts" in reviews.columns:
    biz_match_rate = (
        reviews.groupby("business_id")["checkin_ts"]
        .apply(lambda x: x.notna().mean())
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Per-restaurant match rate distribution
    axes[0].hist(biz_match_rate, bins=30, color="steelblue", edgecolor="white")
    axes[0].axvline(biz_match_rate.median(), color="red", ls="--", label=f"median {biz_match_rate.median():.0%}")
    axes[0].set_title("Checkin Match Rate by Restaurant")
    axes[0].set_xlabel("Match Rate"); axes[0].set_ylabel("Restaurants")
    axes[0].legend()

    # Match gap distribution (how close are the matches?)
    gap_hours = (reviews["date"] - reviews["checkin_ts"]).dt.total_seconds() / 3600
    gap_hours = gap_hours.dropna()

    axes[1].hist(gap_hours.clip(upper=72), bins=72, color="coral", edgecolor="white")
    axes[1].axvline(gap_hours.median(), color="red", ls="--", label=f"median {gap_hours.median():.1f}h")
    axes[1].set_title("Review-to-Checkin Gap (matched only)")
    axes[1].set_xlabel("Hours after checkin"); axes[1].set_ylabel("Reviews")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    # Summary stats
    print(f"Overall match rate: {reviews['checkin_ts'].notna().mean():.1%}")
    print(f"Match rate by restaurant: median {biz_match_rate.median():.0%}, "
          f"mean {biz_match_rate.mean():.0%}")
    print(f"Restaurants with 0% match: {(biz_match_rate == 0).sum():,}")
    print(f"Restaurants with 90%+ match: {(biz_match_rate >= 0.9).sum():,}")
    print(f"\nGap (hours): median {gap_hours.median():.1f}, "
          f"mean {gap_hours.mean():.1f}, 90th pctl {gap_hours.quantile(0.9):.1f}")
else:
    print("No checkin_ts column — run match_checkins_to_reviews first")

### 6. Tips

In [ ]:
if tips is not None:
    tips["text_len"] = tips["text"].str.len()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(tips["text_len"].clip(upper=300), bins=50, color="mediumpurple", edgecolor="white")
    axes[0].set_title("Tip Text Length Distribution")
    axes[0].set_xlabel("Characters (clipped at 300)")

    # Tips per restaurant
    tips_per_biz = tips.groupby("business_id").size()
    axes[1].hist(tips_per_biz.clip(upper=50), bins=50, color="goldenrod", edgecolor="white")
    axes[1].set_title("Tips per Restaurant (clipped at 50)")
    axes[1].set_xlabel("Tip Count")

    plt.tight_layout()
    plt.show()

# Important Stats for Modeling & Validating Our Thresholds

These analyses answer specific "should we build this?" and "did we set that threshold right?" questions before we commit to the Two-Tower architecture and start training.

### Interaction Density Thresholds - k-core analysis

We filter users with <10 reviews and businesses with <5 reviews to ensure enough signal for collaborative filtering. But how sensitive is our dataset to those thresholds? If bumping from 5 to 6 cuts half the businesses, the threshold is fragile and we should be cautious. If the curves flatten around our chosen values, we're in a stable region. This plot shows where the "sweet spot" is — aggressive enough to remove noise, conservative enough to keep data.


In [ ]:
# -- k-core analysis: how many users/items survive at different thresholds --
user_counts = reviews.groupby("user_id").size()
biz_counts = reviews.groupby("business_id").size()

thresholds = range(1, 21)
user_surviving = [int((user_counts >= k).sum()) for k in thresholds]
item_surviving = [int((biz_counts >= k).sum()) for k in thresholds]
interaction_surviving = []
for k in thresholds:
    valid_users = set(user_counts[user_counts >= k].index)
    valid_items = set(biz_counts[biz_counts >= k].index)
    interaction_surviving.append(
        int(reviews[reviews["user_id"].isin(valid_users) & reviews["business_id"].isin(valid_items)].shape[0])
    )

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(thresholds, user_surviving, marker="o", color="teal")
axes[0].set_title("Users Surviving k-Core Filter")
axes[0].set_xlabel("Min Reviews (k)"); axes[0].set_ylabel("Users")

axes[1].plot(thresholds, item_surviving, marker="o", color="salmon")
axes[1].set_title("Items Surviving k-Core Filter")
axes[1].set_xlabel("Min Reviews (k)"); axes[1].set_ylabel("Items")

axes[2].plot(thresholds, interaction_surviving, marker="o", color="mediumpurple")
axes[2].set_title("Interactions Surviving k-Core Filter")
axes[2].set_xlabel("Min Reviews (k)"); axes[2].set_ylabel("Interactions")

plt.tight_layout()
plt.show()

### Temporal Split Viability

We split train/val/test by time (80/10/10 by cumulative review volume), not randomly. Three reasons:

**1. No future leakage.** A random split lets a user's 2019 review land in training while their 2016 review is in test — the model learns from the future to predict the past. A temporal split ensures training < validation < test chronologically, matching real-world inference where we only know a user's history up to "now."

**2. Time-of-visit features.** We use checkin timestamps as context features. A random split would scatter a user's temporal patterns across all three sets, inflating metrics. A temporal split forces the model to generalize forward in time.

**3. Realistic cold start rates.** New users and restaurants appear over time. A temporal split naturally surfaces cold start entities in val/test (users and businesses that didn't exist during training), giving us honest evaluation of the Two-Tower content features. A random split would artificially spread each entity's reviews across sets, hiding the cold start problem.

This plot shows review volume over time with our cutoff dates, confirming the data is stable enough around the split points.

In [ ]:
# -- Review volume by month with candidate split lines --
reviews_monthly = reviews.set_index("date").resample("M").size()

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(reviews_monthly.index, reviews_monthly.values, alpha=0.3, color="steelblue")
ax.plot(reviews_monthly.index, reviews_monthly.values, color="steelblue", linewidth=1)

# Mark 80/10/10 split by cumulative review count
cumulative = reviews_monthly.cumsum()
total = cumulative.iloc[-1]
train_cutoff = cumulative[cumulative >= total * 0.8].index[0]
val_cutoff = cumulative[cumulative >= total * 0.9].index[0]

ax.axvline(train_cutoff, color="red", linestyle="--", label=f"Train cutoff ({train_cutoff.strftime('%Y-%m')})")
ax.axvline(val_cutoff, color="orange", linestyle="--", label=f"Val cutoff ({val_cutoff.strftime('%Y-%m')})")
ax.legend()
ax.set_title("Temporal Split: 80/10/10 by Review Volume")
ax.set_ylabel("Reviews")
plt.tight_layout()
plt.show()

### User-rating variance - can the model learn preferences?

This is a go/no-go check for collaborative filtering. If most users give everything the same rating (low std dev), there's no preference signal to learn — we'd just predict the user's mean and call it a day. A high median std dev means users are differentiating between restaurants they like and don't, which is the signal our Two-Tower model will learn from. Below ~0.5 would be concerning; above ~1.0 means strong, learnable preferences.


In [ ]:
# -- Per-user rating std dev --
user_std = reviews.groupby("user_id")["stars"].agg(["std", "count"])
user_std = user_std[user_std["count"] >= 5]  # only users with enough reviews

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(user_std["std"].dropna(), bins=40, color="teal", edgecolor="white")
ax.axvline(user_std["std"].median(), color="red", linestyle="--",
           label=f"Median σ = {user_std['std'].median():.2f}")
ax.set_title("Per-User Rating Std Dev (users with ≥5 reviews)")
ax.set_xlabel("Std Dev of Star Ratings"); ax.set_ylabel("Users")
ax.legend()
plt.tight_layout()
plt.show()

### Popularity bias - long tail of items

Shows what fraction of restaurants account for most of the reviews. If a small percentage of "head" items dominate, the model will learn great embeddings for popular restaurants but struggle with the long tail. This directly motivates our Two-Tower architecture over pure matrix factorization — the tail restaurants need content features (categories, price tier, location) to get reasonable representations because they don't have enough interactions alone.

In [ ]:
# -- Item popularity CDF --
biz_counts_sorted = reviews.groupby("business_id").size().sort_values(ascending=False)
cumulative_pct = biz_counts_sorted.cumsum() / biz_counts_sorted.sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
item_pct = np.arange(1, len(cumulative_pct) + 1) / len(cumulative_pct) * 100
ax.plot(item_pct, cumulative_pct.values, color="mediumpurple")
ax.axhline(80, color="gray", linestyle=":", alpha=0.5)
pct_at_80 = item_pct[np.searchsorted(cumulative_pct.values, 80)]
ax.axvline(pct_at_80, color="red", linestyle="--",
           label=f"Top {pct_at_80:.0f}% of items → 80% of reviews")
ax.set_title("Item Popularity CDF (Long-Tail Analysis)")
ax.set_xlabel("% of Items (ranked by popularity)"); ax.set_ylabel("% of Total Reviews")
ax.legend()
plt.tight_layout()
plt.show()

### Cold start simulation preview

Cold start entities are users or businesses in val/test that never appeared in training. Pure collaborative filtering (matrix factorization) cannot score these at all — it has no embedding for an unseen ID. The Two-Tower architecture handles this by falling back on content features: categories, price tier, and location for cold restaurants; contextual features like visit time for cold users.

**Val:** 7.3% cold users, 6.4% cold items
**Test:** 10.2% cold users, 11.1% cold items, 3.5% both cold

We evaluate four slices separately:
1. **Warm** — both user and item seen in training. The bulk of traffic; measures core CF quality.
2. **Cold restaurant** — new business, existing user. Can the restaurant tower generalize from categories, price, and location alone?
3. **Cold user** — new user, existing business. Can the user tower generalize from contextual features like visit time and location?
4. **Both cold** — new user and new restaurant (5,989 interactions, 3.5% of test). The strongest test of the Two-Tower architecture — interaction history contributes nothing, so performance here is purely driven by content features in both towers. This is the scenario where Two-Tower should show the clearest advantage over matrix factorization.

The increasing cold rates from val → test are expected with temporal splits, confirming the split is behaving correctly.

In [ ]:
# -- Cold start overlap between temporal splits --
train_reviews = reviews[reviews["date"] <= train_cutoff]
val_reviews = reviews[(reviews["date"] > train_cutoff) & (reviews["date"] <= val_cutoff)]
test_reviews = reviews[reviews["date"] > val_cutoff]

train_users = set(train_reviews["user_id"])
train_items = set(train_reviews["business_id"])

splits = {"Val": val_reviews, "Test": test_reviews}
cold_stats = []
for name, split in splits.items():
    split_users = set(split["user_id"])
    split_items = set(split["business_id"])
    cold_stats.append({
        "Split": name,
        "Cold Users (%)": (1 - len(split_users & train_users) / len(split_users)) * 100,
        "Cold Items (%)": (1 - len(split_items & train_items) / len(split_items)) * 100,
    })

cold_df = pd.DataFrame(cold_stats)
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(cold_df))
w = 0.35
ax.bar(x - w/2, cold_df["Cold Users (%)"], w, label="Cold Users", color="teal")
ax.bar(x + w/2, cold_df["Cold Items (%)"], w, label="Cold Items", color="salmon")
ax.set_xticks(x); ax.set_xticklabels(cold_df["Split"])
ax.set_ylabel("% New (Unseen in Train)")
ax.set_title("Cold Start Rates by Temporal Split")
ax.legend()
for i, row in cold_df.iterrows():
    ax.annotate(f'{row["Cold Users (%)"]:.1f}%', (i - w/2, row["Cold Users (%)"]),
                ha="center", va="bottom", fontsize=9)
    ax.annotate(f'{row["Cold Items (%)"]:.1f}%', (i + w/2, row["Cold Items (%)"]),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
print(f"User rating σ: median {user_std['std'].median():.2f}, mean {user_std['std'].mean():.2f}")
print(f"Long tail: top {pct_at_80:.0f}% of items cover 80% of reviews")
print(f"\nCold start rates:")
print(cold_df.to_string(index=False))

In [ ]:
print(f"States: {restaurants['state'].nunique()} ({', '.join(restaurants['state'].value_counts().index.tolist())})")

In [ ]:
# ── Post-alignment sanity checks ─────────────────────────────────────────
print("=== Dataset Alignment ===")
print(f"Restaurants: {len(restaurants):,}")
print(f"Users:       {len(users):,}")
print(f"Reviews:     {len(reviews):,}")
if checkins is not None:
    print(f"Checkin profiles: {len(checkins):,}")
if tips is not None:
    print(f"Tips:        {len(tips):,}")

# Confirm no orphans
review_biz = set(reviews["business_id"])
review_users = set(reviews["user_id"])
orphan_restaurants = len(set(restaurants["business_id"]) - review_biz)
orphan_users = len(set(users["user_id"]) - review_users)
if checkins is not None:
    orphan_checkins = len(set(checkins["business_id"]) - review_biz)
print(f"\nOrphan restaurants (no reviews): {orphan_restaurants}")
print(f"Orphan users (no reviews):      {orphan_users}")
if checkins is not None:
    print(f"Orphan checkin profiles:         {orphan_checkins}")

# Density thresholds held
user_counts = reviews.groupby("user_id").size()
biz_counts = reviews.groupby("business_id").size()
print(f"\nMin reviews per user:     {user_counts.min()} (threshold: 10)")
print(f"Min reviews per business: {biz_counts.min()} (threshold: 5)")
print(f"Median reviews per user:     {user_counts.median():.0f}")
print(f"Median reviews per business: {biz_counts.median():.0f}")

# Sparsity
n_users = reviews["user_id"].nunique()
n_items = reviews["business_id"].nunique()
density = len(reviews) / (n_users * n_items) * 100
print(f"\nInteraction matrix: {n_users:,} users x {n_items:,} items")
print(f"Density: {density:.4f}%")
print(f"Avg reviews per user: {len(reviews) / n_users:.1f}")
print(f"Avg reviews per item: {len(reviews) / n_items:.1f}")

In [ ]:
# ── Checkin match analysis ────────────────────────────────────────────────
if "checkin_ts" in reviews.columns:
    matched = reviews["checkin_ts"].notna()
    print("=== Checkin-to-Review Matching ===")
    print(f"Overall match rate: {matched.mean():.1%} ({matched.sum():,} / {len(reviews):,})")

    # Match rate by star rating — are matched reviews biased?
    print("\nMatch rate by star rating:")
    print(reviews.groupby("stars")["checkin_ts"].apply(lambda x: f"{x.notna().mean():.1%}").to_string())

    # Match rate by year — temporal bias?
    reviews["year"] = reviews["date"].dt.year
    print("\nMatch rate by year:")
    print(reviews.groupby("year")["checkin_ts"].apply(lambda x: f"{x.notna().mean():.1%}").to_string())

    # Per-business match rate distribution
    biz_match = reviews.groupby("business_id")["checkin_ts"].apply(lambda x: x.notna().mean())
    print(f"\nPer-business match rate:")
    print(f"  Median: {biz_match.median():.1%}")
    print(f"  Mean:   {biz_match.mean():.1%}")
    print(f"  Businesses with 0% match:   {(biz_match == 0).sum():,} ({(biz_match == 0).mean():.1%})")
    print(f"  Businesses with >50% match: {(biz_match > 0.5).sum():,} ({(biz_match > 0.5).mean():.1%})")

    # Key question: does the model even need checkin_ts per review,
    # or is the business-level checkin profile sufficient?
    print(f"\nBusinesses with checkin profiles: {len(checkins):,}")
    print(f"Businesses in reviews:           {reviews['business_id'].nunique():,}")
    if checkins is not None:
        profile_coverage = reviews["business_id"].isin(set(checkins["business_id"])).mean()
        print(f"Reviews at businesses WITH profiles: {profile_coverage:.1%}")

    reviews.drop(columns=["year"], inplace=True)

In [ ]:
# Cross-tab: how many cities per state?
print("Cities per state:")
for state, group in restaurants.groupby("state"):
    cities = group["city"].value_counts()
    total = len(group)
    print(f"\n  {state} ({total:,} restaurants, {len(cities)} cities):")
    for city, count in cities.head(10).items():
        print(f"    {city}: {count:,} ({count/total*100:.1f}%)")

In [ ]:
train = pd.read_parquet("../data/splits/train.parquet")
print(train.columns.tolist())
print(train.shape)

print(restaurants.columns.tolist())
print(restaurants.shape)

In [ ]:
print(f"checkin_ts non-null: {train['checkin_ts'].notna().sum()} / {len(train)} ({train['checkin_ts'].notna().mean():.1%})")

In [ ]:
city_counts = restaurants.groupby(["state", "city"]).size().reset_index(name="n_restaurants")
print(f"Total cities: {len(city_counts)}")
print(f"\nDistribution of restaurants per city:")
print(city_counts["n_restaurants"].describe())
print(f"\nCities with < 5 restaurants: {(city_counts['n_restaurants'] < 5).sum()}")
print(f"Cities with < 10 restaurants: {(city_counts['n_restaurants'] < 10).sum()}")
print(f"\nTop 20 cities:")
print(city_counts.sort_values("n_restaurants", ascending=False).head(20).to_string(index=False))

In [ ]:
for min_pool in [5, 10, 20, 50]:
    valid_cities = city_counts[city_counts["n_restaurants"] >= min_pool]
    valid = restaurants[restaurants["city"].isin(valid_cities["city"])]
    valid_biz = set(valid["business_id"])
    reviews_kept = train_reviews[train_reviews["business_id"].isin(valid_biz)]
    print(f"min_pool={min_pool:3d} → {len(valid_cities):4d} cities, "
          f"{len(valid):,} restaurants, {len(reviews_kept):,} reviews "
          f"({len(reviews_kept)/len(train_reviews)*100:.1f}%)")

In [ ]:
users = pd.read_parquet("../data/raw/extracted/user.parquet")
reviews = pd.read_parquet("../data/processed/reviews.parquet")

print(f"Total Yelp users:                    {len(users):,}")
print(f"Users who reviewed a restaurant:     {reviews['user_id'].nunique():,}")
print(f"  ... with 10+ restaurant reviews:   {(reviews.groupby('user_id').size() >= 10).sum():,}")